# Fraud Detection MLOps — narrativa técnica

Autor: Wanderson Ferreira. Marco documentado: 2026-10-06.

Este notebook reúne sínteses, decisões e resultados que mudam a história do
projeto. Investigação detalhada fica nos notebooks por etapa; comandos e
procedimentos ficam em `docs/`, conforme a [política](../docs/DOCUMENTATION_POLICY.md).


## Problema e direção

Construir um projeto de classificação para detecção de fraude com uma base
temporal reproduzível. A evolução pretendida inclui avaliação temporal e operação
do modelo, com replay/streaming e observabilidade definidos por etapa.

A fonte é simulada. Não há ainda métrica de modelo, objetivo quantitativo de
negócio validado nem evidência de operação real. A [síntese para stakeholders](../docs/STAKEHOLDERS.md)
distingue as entregas atuais dos próximos marcos.


## Marco 1 — Bronze reproduzível

Implementação: [`18242fc`](https://github.com/wanderson42/fraud-detection-mlops/commit/18242fcc8e340bad52a394c7c3624b78bd809b6c).

Decisão central: fixar uma revisão do Handbook e preservar seus bytes originais,
com inventário, checksums, manifesto e auditorias. Isso estabelece uma base
rastreável para a futura transformação ELT.

O autor informou a aquisição completa e a verificação de **183/183 arquivos**,
cobrindo 2018-04-01 a 2018-09-30. A execução completa reutilizou sete arquivos e
baixou os 176 restantes. Foram informados **39 testes locais aprovados**; a
[CI dessa revisão](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37514553686)
foi consultada e está aprovada.

Naquele marco, a validação cobria aquisição e integridade; o diagnóstico semântico
foi realizado na etapa seguinte.
Fontes: [recibo](../references/evidence/bronze_2026-10-06.json) e
[notebook da Bronze](stages/01_bronze_ingestion.ipynb).


## Marco 2 — diagnóstico completo e contrato da Silver

O autor informou o diagnóstico dos 183 arquivos: **1.754.155 transações**, com o
mesmo número de IDs distintos, 14.681 fraudes e 1.739.474 transações genuínas.
Os controles implementados não apontaram nulos, duplicatas ou inconsistências
temporais/de rótulos. Foram encontrados **42 valores monetários zero**.
O [recibo](../references/evidence/silver_profile_2026-10-06.json) transcreve a saída
do terminal; não é um output de execução deste notebook.

Decisão: preservar todos os registros, inclusive zeros e extremos, e converter
as nove colunas para tipos explícitos. A política se apoia no diagnóstico e no
arredondamento do gerador publicado. Consulte o [contrato](../docs/SILVER_CONTRACT.md)
e o [notebook da Silver](stages/02_silver_data_contract.ipynb).

O autor também concluiu a construção e a verificação da **Silver `silver_v1`**:
183 partições Parquet, com todas as contagens reconciliadas e os 42 zeros
preservados. Foram informados 70 testes locais aprovados em 0,91 s. A execução
responsável é `f9fc5a5083f14d9fadf6892a531e1488`, registrada no
[recibo da Silver](../references/evidence/silver_build_2026-10-06.json).

Os resultados são saídas de terminal fornecidas pelo autor; os arquivos nativos
permanecem locais. A implementação foi publicada em `acf1516`, com
[CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37563463479).
A próxima etapa usa a Silver para EDA e definição do protocolo temporal; a Gold
conterá as features, com testes contra vazamento.


## Validação de código — integração tox–Poetry

A configuração [tox.toml](../tox.toml) centraliza a sequência usada localmente e
na CI: verificar o lockfile, sincronizar dependências e executar Ruff/pytest em
`.tox/py314`. O Poetry instala as versões fixadas do lockfile nesse ambiente.
Uma checagem anterior à sincronização confirma o alvo e evita operar sobre a
`.venv` de desenvolvimento.

A integração foi validada no ambiente de preparação; a execução local do autor
e a CI desta alteração ainda aguardam aplicação/publicação. O escopo é o projeto
em modo editable a partir do checkout; um teste de wheel fora do checkout é um
contrato diferente. O processo completo está em [Testing](../docs/TESTING.md).
Não há alteração dos dados ou resultados de modelo nessa melhoria.


## EDA da Silver e protocolo temporal — preparação de 2026-10-07

O autor informou validação local do tox: 70 testes em 1,57 s, com lint e formatação aprovados;
a execução completa levou 4,61 s. A nova CI dessa integração ainda não foi informada.

A [EDA](../docs/EDA.md) preparada usa somente o treino de 1 a 28 de abril.
O [protocolo temporal](../docs/EVALUATION_PROTOCOL.md) reserva validação e teste final,
com sete dias de feedback entre as janelas. O teste final permanece fora da seleção inicial;
a futura Gold implementará as features e os splits.

O [notebook da etapa](stages/03_silver_eda.ipynb) orienta a leitura dos relatórios locais.
A execução real e seus achados permanecem pendentes; o
[recibo](../references/evidence/eda_preparation_2026-10-07.json) registra apenas a preparação.
Não há modelo avaliado nem nova métrica de detecção.


## EDA de treino informada pelo autor — 2026-10-07

Execução `733686de23204cd6b9a5a1ec1cfbc2e7`: nove outputs verificados, 268.668 transações,
1.505 fraudes (0,5602%), quatro valores zero preservados e IDs distintos.
A evidência é [saída de terminal compartilhada](../references/evidence/eda_training_2026-10-07.json).
O autor compartilhou quatro tabelas agregadas; os arquivos nativos e o gráfico não foram inspecionados diretamente.

> O desbalanceamento já mostra por que **acurácia não será nossa métrica principal**: prever todas as transações como genuínas produziria aproximadamente **99,44% de acurácia**, com **recall de fraude igual a zero**. Isso sustenta a escolha de Average Precision para avaliar o ranking.

O cálculo é um contrafactual da classe majoritária no treino, sem modelo ajustado nem uso de holdouts.
A prevalência cresce entre os quatro blocos semanais; os valores de fraude apresentam cauda maior,
com ocorrências de valor baixo. A hipótese inicial para Gold é combinar valor atual e históricos
causais de cliente/terminal, tratando histórico insuficiente e atraso dos rótulos.

Os [achados e hipóteses](../docs/EDA.md#achados-informados-pelo-autor-em-2026-10-07)
estão ligados ao [notebook da etapa](stages/03_silver_eda.ipynb).
Commit do código avaliado, checks locais da EDA e CI correspondente ainda não foram informados.
Não há modelo avaliado; a Gold é a próxima implementação.


## Gold causal — preparação de 2026-10-07

A EDA e a integração tox foram publicadas em `fc22a48`, com
[CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37634000505).
A [Gold](../docs/GOLD_CONTRACT.md) preparada materializa 19 preditores nas três janelas
do protocolo, preservando linhas e separando alvo/IDs dos preditores.

Históricos de cliente/terminal usam passado estrito de 1 e 7 dias. Rótulos conhecidos
usam janelas deslocadas em sete dias; peers são excluídos e históricos vazios são tratados
com suporte explícito e flags de validade. Não há transformações aprendidas nesta etapa.

O [notebook da Gold](stages/04_gold_temporal_features.ipynb) e o
[recibo](../references/evidence/gold_preparation_2026-10-07.json) descrevem decisões e testes
controlados. Construção real e CI da Gold estão pendentes; ainda não há modelo avaliado.


## Gold real — execução informada em 2026-10-07

O autor forneceu build com status `success` e verify concordante: **42 partições,
19 preditores e 402.877 linhas**. Treino: 268.668; validação: 67.255; teste: 66.954.
O treino reconcilia com a EDA anterior. Run de auditoria: `8b1564df49fc497080963986b05aa007`.

O [recibo da execução](../references/evidence/gold_build_2026-10-07.json)
transcreve o terminal e distingue essa evidência dos testes controlados. Os Parquets,
o manifesto e a auditoria nativos não foram inspecionados pelo assistente. Revisão `14ab57e` e CI aprovada foram conferidas.
A saída dos checks locais do autor ainda não foi informada. Os outputs salvos das células de verificação e leitura do treino
foram publicados pelo autor e preservados nesta atualização; o assistente não reexecutou o notebook.

Os 19 preditores são candidatos escolhidos no desenho do contrato. A seleção por
desempenho ficará no pipeline: ajuste no treino, comparação na validação e teste
final após congelar as escolhas. Não há modelo avaliado neste marco.


## Publicação e CI conferidas em 2026-10-07

Revisão [`14ab57e`](https://github.com/wanderson42/fraud-detection-mlops/commit/14ab57e15f0931ffb6966b26d390a42b514762b7),
com [CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37652509812):
104 testes em 11,37 s; lint, formatação e lockfile aprovados.
O [recibo](../references/evidence/gold_build_2026-10-07.json) associa
a construção real informada pelo autor à revisão publicada e à evidência da CI.

O notebook da Gold publicado contém verificação concordante e leitura do treino:
`X` (268.668, 19), `y` (268.668,) e metadados (268.668, 6). Seus outputs foram
conferidos e preservados; não foram gerados pela preparação desta documentação.
A CI usa dados controlados, e os artefatos nativos da Gold continuam locais.
Nenhum modelo foi avaliado. O próximo marco é o baseline temporal.
